#Transformation
###Using SQL query inside pyspark

In [0]:
query = """
select 
    row_number() over (order by ci.customer_id) as customer_key,
    ci.customer_id, ci.customer_key as source_customer_key, ci.customer_firstname, ci.customer_lastname, la.country, ci.customer_marital_status,
    case 
        when ci.customer_gender <> 'n/a' then ci.customer_gender
        else coalesce(ca.gender, 'n/a')
    end as gender,
    ca.dob as birth_date,
    ci.customer_create_date as create_date
from workspace.silver.crm_customers ci
left join workspace.silver.erp_customers ca on ci.customer_key = ca.cust_id
left join workspace.silver.erp_customer_location la on ci.customer_key = la.customer_number
"""

df = spark.sql(query)

In [0]:
display(df)

This query creates a Gold customer dimension table by combining customer information from three Silver tables. silver.crm_customers is used as the main/base table and is given the alias ci; it is then LEFT JOINED with silver.erp_customers (ca) and silver.erp_customer_location (la) using customer_number, so all CRM customers are retained even if matching ERP information is missing. ROW_NUMBER() OVER (ORDER BY ci.customer_id) generates a sequential surrogate key called customer_key. The query then selects customer details such as customer_id, customer_number, first_name, last_name, and marital_status from CRM, while country comes from the ERP location table and birth_date comes from the ERP customer table. For gender, a CASE expression first checks whether the CRM gender is not 'n/a'; if it is valid, CRM gender is used, otherwise COALESCE(ca.gender, 'n/a') uses the ERP gender if available, and 'n/a' if both sources are unavailable. Finally, birth_date and created_date are renamed to birthdate and create_date using AS. The overall purpose is to create a single, enriched, standardized customer table from multiple Silver sources so downstream users do not have to repeatedly perform these joins and data-cleaning rules.

#Writing Gold table

In [0]:

df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_customers")